In [2]:
!pip install pyswip

In [4]:
!apt-get install swi-prolog

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  gettext gettext-base intltool-debian libarchive-cpio-perl
  libarchive-zip-perl libdebhelper-perl libfile-stripnondeterminism-perl
  libmail-sendmail-perl libossp-uuid16 libsub-override-perl
  libsys-hostname-long-perl libtool po-debconf swi-prolog-core
  swi-prolog-core-packages swi-prolog-doc swi-prolog-nox swi-prolog-x
Suggested packages:
  dh-make gettext-doc libasprintf-dev libgettextpo-dev uuid libtool-doc
  gcj-jdk libmail-box-perl elpa-ediprolog swi-prolog-java swi-prolog-odbc
  swi-prolog-bdb
The following NEW packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  gettext gettext-base intltool-debian libarchive-cpio-perl
  libarchive-zip-perl libdebhelper-perl libfile-stripnondeterminism-perl
  libmail-send

In [7]:
import pandas as pd
from pyswip import Prolog
from sklearn.linear_model import LogisticRegression

# =====================================================================
# PASSO 1: Criar o arquivo Prolog (rede_social.pl) sem caracteres ocultos
# =====================================================================
conteudo_prolog = """
% Fatos do grafo social
transacao_entre(joao, ana, 1500).
transacao_entre(ana, carlos, 800).
transacao_entre(carlos, daniel, 50).

% Historico de inadimplencia
inadimplente(daniel).

% Caso Base: Conexao direta (Grau 1) em ambas as direcoes
risco_conexao(X, Y, 1) :- transacao_entre(X, Y, _).
risco_conexao(X, Y, 1) :- transacao_entre(Y, X, _).

% Caso Recursivo: Calcula a distancia acumulando os graus
risco_conexao(X, Y, Grau) :-
    transacao_entre(X, Z, _),
    X \\== Y,
    risco_conexao(Z, Y, GrauAnterior),
    Grau is GrauAnterior + 1.
"""

# Salva o arquivo .pl no diretorio local automaticamente
with open("rede_social.pl", "w") as f:
    f.write(conteudo_prolog)


# =====================================================================
# PASSO 2: Criar o arquivo de dados financeiros tradicionais (CSV)
# =====================================================================
conteudo_csv = """cliente_id,renda_mensal,score_classico,inadimplente_historico
joao,5200,750,0
ana,3100,610,0
carlos,1800,420,1"""

# Salva o arquivo .csv no diretorio local automaticamente
with open("dados_financeiros.csv", "w") as f:
    f.write(conteudo_csv)


# =====================================================================
# PASSO 3: Inicializar o Motor do Prolog e Consultar a Base Lógica
# =====================================================================
prolog = Prolog()
prolog.consult("rede_social.pl")


# =====================================================================
# PASSO 4: Pipeline Python (Extração de Features e Treinamento)
# =====================================================================
# Carrega os dados tradicionais para o Pandas DataFrame
df = pd.read_csv("dados_financeiros.csv")

# Funcao de ponte que consulta o Prolog para extrair a feature de rede
def obter_grau_risco(nome):
    # Formata a query de forma simples e direta para evitar erros de sintaxe no pyswip
    query_string = "risco_conexao(" + str(nome) + ", daniel, Grau)"
    query = list(prolog.query(query_string))

    if query:
        return query[0]["Grau"]
    return 999  # Retorna um valor alto caso nao haja conexao com a entidade de risco

# Aplica a funcao e cria a nova coluna estruturada no DataFrame
df['grau_risco_rede'] = df['cliente_id'].apply(obter_grau_risco)

# Separa as features (X) e a variável alvo (y)
X = df[['renda_mensal', 'score_classico', 'grau_risco_rede']]
y = df['inadimplente_historico']

# Treina o modelo estatistico de Regressao Logistica
modelo = LogisticRegression()
modelo.fit(X, y)


# =====================================================================
# PASSO 5: Exibição dos Resultados (Inferência Explicável)
# =====================================================================
print("--- PIPELINE EXECUTADO COM SUCESSO ---")
print("Pesos das Features (Renda, Score, Grau Risco):", modelo.coef_)
print("Intercepto do Modelo:", modelo.intercept_)

print("\n--- Exemplo de Saída Probabilística Estilo ProbLog ---")
# Simulando a predicao de um novo cliente com renda 2500, score 500 e a 2 graus de distancia do daniel
novo_cliente = [[2500, 500, 2]]
probabilidade = modelo.predict_proba(novo_cliente)[0][1]
print(f"{probabilidade:.2f} :: risco(cliente_novo) :- conectado_a(cliente_novo, daniel, 2).")

--- PIPELINE EXECUTADO COM SUCESSO ---
Pesos das Features (Renda, Score, Grau Risco): [[-1.69430254e-02 -2.47733393e-03 -1.19686290e-05]]
Intercepto do Modelo: [42.78552615]

--- Exemplo de Saída Probabilística Estilo ProbLog ---
0.31 :: risco(cliente_novo) :- conectado_a(cliente_novo, daniel, 2).


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [8]:
import pandas as pd
from pyswip import Prolog
from sklearn.linear_model import LogisticRegression

# 1. Inicializar e conectar à base lógica do Prolog
prolog = Prolog()
prolog.consult("rede_social.pl")  # Carrega o arquivo feito na Parte 1

# 2. Carregar o dataset financeiro tradicional com Pandas
df = pd.read_csv("dados_financeiros.csv")

# 3. Função de extração de features lógicas (A Ponte)
def obter_grau_risco(nome):
    # Executa a consulta dinâmica no motor Prolog em relação ao alvo 'daniel'
    query = list(prolog.query(f"risco_conexao({nome}, daniel, Grau)"))

    if query:
        # Extrai o valor associado à variável unificada 'Grau'
        return query[0]["Grau"]
    else:
        # Penalidade padrão caso não haja nenhuma conexão identificada no grafo
        return 999

# Aplica a função de forma vetorizada criando a nova feature relacional
df['grau_risco_rede'] = df['cliente_id'].apply(obter_grau_risco)

# 4. Preparação dos dados para o modelo estatístico
# Recursos (X): Combinação de dados numéricos clássicos e a estrutura de rede do Prolog
X = df[['renda_mensal', 'score_classico', 'grau_risco_rede']]
y = df['inadimplente_historico']

# 5. Treinamento da Regressão Logística (Calibração Numérica)
modelo = LogisticRegression()
modelo.fit(X, y)

print("--- Treinamento Concluído ---")
print("Coeficientes Aprendidos (Pesos):", modelo.coef_)
print("Intercepto:", modelo.intercept_)

--- Treinamento Concluído ---
Coeficientes Aprendidos (Pesos): [[-1.69430254e-02 -2.47733393e-03 -1.19686290e-05]]
Intercepto: [42.78552615]
